<a href="https://colab.research.google.com/github/eeeewyz/Audio-course/blob/main/ASR/Eval_base_whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#下载whisper的pipline
from transformers import pipeline
import torch

if torch.cuda.is_available():
    device = "cuda:0"
    torch_dtype = torch.float16
else:
    device = "cpu"
    torch_dtype = torch.float32

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    torch_dtype=torch_dtype,
    device=device,
)

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

In [3]:
from datasets import load_dataset

#加载数据集
ds =load_dataset("ylacombe/english_dialects", "irish_male", split="train")



In [5]:
# 看下数据集属性
ds.column_names

['line_id', 'audio', 'text', 'speaker_id']

In [6]:
from transformers.pipelines.pt_utils import KeyDataset

# 取前 3 条样本
small_dataset = ds.select(range(5))

#在pipeline做预测
results = pipe(
    KeyDataset(small_dataset, "audio"),
    generate_kwargs={"task": "transcribe"},
    batch_size=1
)

all_predictions = [result["text"] for result in results]

#打印prediction
all_predictions

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[' It is 13 degrees with drizzle and Exeter.',
 ' In one version of this technique, a conductive sheet under test is placed between two coils.',
 ' People lock but no one ever finds it.',
 ' Throughout the centuries, people have explained the rainbow in various ways.',
 " The powers they appoint to chair are limited so the chair can't adjourn a meeting at any point without the majority vote."]

In [9]:
#用wer做预测
!pip install --upgrade evaluate jiwer
from evaluate import load

wer_metric = load("wer")



wer = 100 * wer_metric.compute(
    references=small_dataset["text"], predictions=all_predictions
)
wer

21.21212121212121

In [11]:
#准备做normalization
#先下载normalizer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()

In [12]:
#进行noamaliz
all_predictions_norm = [normalizer(pred) for pred in all_predictions]
all_references_norm = [normalizer(label) for label in small_dataset["text"]]

In [13]:
# 先过滤掉 reference 为空的样本，再计算 WER。
# all_predictions_norm = [
#     all_predictions_norm[i]
#     for i in range(len(all_predictions_norm))
#     if len(all_references_norm[i]) > 0

# predictions = ["hello world", "abc", "good morning"]
# references  = ["hello word",  "",    "good morning"]
# 过滤后变成：
# predictions = ["hello world", "good morning"]
# references  = ["hello word",  "good morning"]


In [14]:
# filtering step to only evaluate the samples that correspond to non-zero references
all_predictions_norm = [
    all_predictions_norm[i]
    for i in range(len(all_predictions_norm))
    if len(all_references_norm[i]) > 0
]
all_references_norm = [
    all_references_norm[i]
    for i in range(len(all_references_norm))
    if len(all_references_norm[i]) > 0
]

wer = 100 * wer_metric.compute(
    references=all_references_norm, predictions=all_predictions_norm
)

wer

10.44776119402985

In [ ]:
#可发现比刚才wer下降了